# components.table

> Table layout rendering: header row, data rows, and cells using CSS Grid.

In [ ]:
#| default_exp components.table

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Optional, Tuple

from fasthtml.common import Div, Span

from cjm_fasthtml_tailwind.utilities.spacing import p
from cjm_fasthtml_tailwind.utilities.typography import font_weight, truncate, font_size
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import grid_display, items
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.sizing import h, min_h
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_daisyui.utilities.semantic_colors import border_dui, bg_dui, text_dui

from cjm_fasthtml_virtual_collection.core.models import (
    ColumnDef, VirtualCollectionConfig, VirtualCollectionState,
    CellRenderContext, RowRenderContext,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

## Grid Template Helper

In [ ]:
#| export
def grid_template_columns(columns: Tuple[ColumnDef, ...],  # Column definitions
                         ) -> str:  # CSS grid-template-columns value
    """Build CSS grid-template-columns from column definitions."""
    return " ".join(col.width for col in columns)

## Header Row

In [ ]:
#| export
def render_header_cell(column: ColumnDef,  # Column definition
                      ) -> Div:  # Header cell element
    """Render a single header cell."""
    cls = combine_classes(
        p.x(2), p.y(1),
        font_weight.semibold, font_size.sm,
        truncate,
        column.header_cls or None,
    )
    return Div(Span(column.header) if column.header else "", cls=cls)

In [ ]:
#| export
def render_header_row(config: VirtualCollectionConfig,  # Collection config
                      ids: VirtualCollectionHtmlIds,     # HTML IDs
                     ) -> Div:  # Header row element
    """Render the table header row."""
    gtc = grid_template_columns(config.columns)
    cells = [render_header_cell(col) for col in config.columns]
    return Div(
        *cells,
        id=ids.header,
        cls=combine_classes(grid_display, items.center, border.b(), border_dui.base_200),
        style=f"grid-template-columns: {gtc}",
    )

## Data Rows

In [ ]:
#| export
def render_data_cell(item: Any,                    # Data item
                     column: ColumnDef,             # Column definition
                     row_index: int,                # Row index
                     total_items: int,              # Total item count
                     ids: VirtualCollectionHtmlIds,  # HTML IDs
                     render_cell: Callable,          # Consumer cell render callback
                     is_cursor: bool = False,        # Whether row is keyboard cursor
                    ) -> Div:  # Cell element with stable ID
    """Render a single data cell with a stable ID for OOB updates."""
    ctx = CellRenderContext(
        row_index=row_index,
        column=column,
        total_items=total_items,
        is_cursor=is_cursor,
    )
    content = render_cell(item, ctx)
    cls = combine_classes(
        p.x(2), p.y(1),
        column.cell_cls or None,
    )
    return Div(
        content,
        id=ids.cell_id(row_index, column.key),
        cls=cls,
    )

In [ ]:
#| export
def render_data_row(item: Any,                       # Data item
                    row_index: int,                   # Row index in full collection
                    config: VirtualCollectionConfig,   # Collection config
                    state: VirtualCollectionState,     # Collection state
                    ids: VirtualCollectionHtmlIds,     # HTML IDs
                    render_cell: Callable,             # Consumer cell render callback
                   ) -> Div:  # Row element with stable ID
    """Render a single data row with all cells."""
    gtc = grid_template_columns(config.columns)
    is_cursor = (row_index == state.cursor_index)

    cells = [
        render_data_cell(
            item, col, row_index, state.total_items, ids, render_cell, is_cursor
        )
        for col in config.columns
    ]

    row_cls = combine_classes(
        grid_display, items.center,
        border.b(), border_dui.base_200,
        bg_dui.primary.opacity(10) if is_cursor else None,
    )

    return Div(
        *cells,
        id=ids.row_id(row_index),
        cls=row_cls,
        style=f"grid-template-columns: {gtc}; height: {config.row_height}px",
    )

## Slots

In [ ]:
#| export
def render_slot(
    slot_index: int,                       # Position in viewport (0-based)
    item: Any,                             # Data item
    item_index: int,                       # Row index in full collection
    config: VirtualCollectionConfig,       # Collection config
    state: VirtualCollectionState,         # Collection state
    ids: VirtualCollectionHtmlIds,         # HTML IDs
    render_cell: Callable,                 # Consumer cell render callback
    oob: bool = False,                     # Whether to render as OOB swap
) -> Div:  # Slot wrapper containing the data row
    """Render a viewport slot wrapping a data row."""
    inner_row = render_data_row(item, item_index, config, state, ids, render_cell)
    return Div(
        inner_row,
        id=ids.slot_id(slot_index),
        hx_swap_oob="innerHTML" if oob else None,
    )

In [ ]:
#| export
def render_table_rows(items: list,                       # Full item list
                      config: VirtualCollectionConfig,    # Collection config
                      state: VirtualCollectionState,      # Collection state
                      ids: VirtualCollectionHtmlIds,      # HTML IDs
                      render_cell: Callable,              # Consumer cell render callback
                     ) -> Div:  # Rows container with slot wrappers
    """Render all visible rows in the current window, wrapped in position-based slots."""
    from cjm_fasthtml_virtual_collection.core.windowing import compute_window

    render_start, render_end = compute_window(
        state.window_start, state.visible_rows, state.total_items
    )

    slots = [
        render_slot(
            slot_index=slot_idx, item=items[item_idx], item_index=item_idx,
            config=config, state=state, ids=ids, render_cell=render_cell,
        )
        for slot_idx, item_idx in enumerate(range(render_start, render_end))
    ]

    return Div(*slots, id=ids.rows)

## Tests

## OOB Helpers

In [ ]:
#| export
def render_cell_oob(item: Any,                       # Data item
                    column: ColumnDef,                # Column to render
                    row_index: int,                   # Row index
                    total_items: int,                 # Total item count
                    ids: VirtualCollectionHtmlIds,    # HTML IDs
                    render_cell: Callable,            # Consumer cell render callback
                    is_cursor: bool = False,          # Whether row is keyboard cursor
                   ) -> Div:  # Cell element with hx-swap-oob
    """Render a single cell with OOB swap for targeted update."""
    cell = render_data_cell(item, column, row_index, total_items, ids, render_cell, is_cursor)
    cell.attrs["hx-swap-oob"] = "outerHTML"
    return cell


def render_row_oob(item: Any,                       # Data item
                   row_index: int,                   # Row index
                   config: VirtualCollectionConfig,  # Collection config
                   state: VirtualCollectionState,    # Collection state
                   ids: VirtualCollectionHtmlIds,    # HTML IDs
                   render_cell: Callable,            # Consumer cell render callback
                  ) -> Div:  # Row element with hx-swap-oob
    """Render a full row with OOB swap for targeted update."""
    row = render_data_row(item, row_index, config, state, ids, render_cell)
    row.attrs["hx-swap-oob"] = "outerHTML"
    return row

In [ ]:
from fasthtml.common import to_xml
from cjm_fasthtml_virtual_collection.core.models import ColumnDef, VirtualCollectionConfig, VirtualCollectionState
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

# Test setup
columns = (
    ColumnDef(key="name", header="Name", width="1fr"),
    ColumnDef(key="size", header="Size", width="100px"),
)
config = VirtualCollectionConfig(prefix="t", columns=columns, row_height=40)
state = VirtualCollectionState(total_items=5, visible_rows=3)
ids = VirtualCollectionHtmlIds(prefix=config.prefix)

In [ ]:
# Test grid_template_columns
assert grid_template_columns(columns) == "1fr 100px"
print(f"Grid template: {grid_template_columns(columns)}")

Grid template: 1fr 100px


In [ ]:
# Test render_header_row
header = render_header_row(config, ids)
header_html = to_xml(header)
assert 'id="t-header"' in header_html
assert 'grid-template-columns: 1fr 100px' in header_html
assert 'Name' in header_html
assert 'Size' in header_html
print("Header row test passed")

Header row test passed


In [ ]:
# Test render_data_row
from dataclasses import dataclass

@dataclass
class Item:
    name: str
    size: str

test_items = [Item(f"file_{i}.txt", f"{i}KB") for i in range(5)]

def test_render_cell(item, ctx):
    if ctx.column.key == "name": return Span(item.name)
    if ctx.column.key == "size": return Span(item.size)

row = render_data_row(test_items[2], 2, config, state, ids, test_render_cell)
row_html = to_xml(row)
assert 'id="t-row-2"' in row_html
assert 'id="t-row-2-col-name"' in row_html
assert 'id="t-row-2-col-size"' in row_html
assert 'file_2.txt' in row_html
assert 'height: 40px' in row_html
print("Data row test passed")

Data row test passed


In [ ]:
# Test render_table_rows — slots wrap rows
rows_container = render_table_rows(test_items, config, state, ids, test_render_cell)
rows_html = to_xml(rows_container)
assert 'id="t-rows"' in rows_html
# Slot wrappers present
assert 'id="t-slot-0"' in rows_html
assert 'id="t-slot-1"' in rows_html
assert 'id="t-slot-2"' in rows_html
assert 'id="t-slot-3"' not in rows_html  # visible_rows=3
# Data rows inside slots
assert 'id="t-row-0"' in rows_html
assert 'id="t-row-1"' in rows_html
assert 'id="t-row-2"' in rows_html
assert 'id="t-row-3"' not in rows_html
print("Table rows (with slots) test passed")

In [ ]:
# Test render_cell_oob
col_name = columns[0]  # "name" column
cell_oob = render_cell_oob(test_items[1], col_name, 1, 5, ids, test_render_cell)
cell_html = to_xml(cell_oob)
assert 'id="t-row-1-col-name"' in cell_html
assert 'hx-swap-oob="outerHTML"' in cell_html
assert 'file_1.txt' in cell_html
print("render_cell_oob test passed")

# Test render_row_oob
row_oob = render_row_oob(test_items[3], 3, config, state, ids, test_render_cell)
row_html = to_xml(row_oob)
assert 'id="t-row-3"' in row_html
assert 'hx-swap-oob="outerHTML"' in row_html
assert 'file_3.txt' in row_html
assert 'id="t-row-3-col-name"' in row_html
assert 'id="t-row-3-col-size"' in row_html
print("render_row_oob test passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()